---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: You Can Just Build Things

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our firstfour lectures, we've covered how
1. We can call LLMs via APIs and get structured responses
2. We can build lexical search with BM25
3. We can build semantic search with embeddings
4. We can combine lexical and semantic search into hybrid search

Today you will put it all together by building a Retrieval Augmented Generation (RAG) system.
- This is a question-answering bot that can answer questions about Fordham University
- You will use real data scraped from the Fordham website.


Your RAG pipeline will look like this:

```
User Question
     ↓
1. RETRIEVE: Find relevant documents (search!)
     ↓
2. AUGMENT: Stuff those documents into a prompt
     ↓
3. GENERATE: Ask an LLM to answer using the context
     ↓
Answer
```


---

# 1. Look at your data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more.

Your task: **look at the data**
- The first step in any AI engineering or data science project should always be to familiarize yourself with the data.
- I cannot stress this enough.. without this step, it's hard to build anything useful.

Tips:
- Unzip the archive and look at some of the files. 
- Open a few in a text editor. 
- Get a feel for what you're working with.
- The first line of every file is always the **URL** of the page it was scraped from. The rest is the page content converted to Markdown. Here's an example — `gabelli-school-of-business_veterans.md`:

```markdown
https://www.fordham.edu/gabelli-school-of-business/veterans

# Military Veterans & Active Duty Members of the Military

## Transform Your Knowledge & Skills Into a Business Career for the Future

As a veteran or an active duty member of the United States Armed Services,
you have gained or are currently acquiring the invaluable organizational,
leadership, analytics, and technical knowledge and skills that hiring
managers seek. These transferrable skills provide a major advantage in
emerging, business-related industries where innovation, a global mind-set,
and the ability to lead individuals and teams in the continuously evolving
work environment, are critical for success.

By completing a graduate or undergraduate business degree at the Gabelli
School of Business, you can prepare for a lifelong career in some of
today's fastest-growing fields. ...

### Study at a Top-Ranked, Military-Friendly University

The Gabelli School of Business is part of Fordham University, the only
New York City university to be among those ranked "Best for Vets" by
Military Times. ...

### Learn How the Yellow Ribbon Program Works

The Yellow Ribbon GI Education Enhancement Program, or the Yellow Ribbon
Program, is a part of the Post-9/11 Veterans Educational Assistance Act
of 2008. ...
```

The filenames mirror the URL structure — underscores replace path separators (e.g. `gabelli-school-of-business_veterans.md` came from `/gabelli-school-of-business/veterans`). Some files are short (a few lines), others are quite long.

- Once you've looked around, load the files into Python. Python's built-in `zipfile` module can read zip archives without extracting to disk. Load them into a list of dictionaries or a DataFrame with at least two fields: the filename (or a clean page name) and the content

In [4]:
# Placeholder for your implementation
import os
import pandas as pd

folder_path = "/Users/purtilalan/Downloads/fordham-website"

documents = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".md"):   # important
            filepath = os.path.join(root, file)
            
            with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
                
                lines = content.split("\n", 1)
                url = lines[0].strip()
                body = lines[1] if len(lines) > 1 else ""
                
                documents.append({
                    "filename": file,
                    "url": url,
                    "content": body
                })

df = pd.DataFrame(documents)

print(f"Loaded {len(df)} documents")


Loaded 9530 documents


In [5]:
import os
os.listdir("/Users/purtilalan/Downloads/fordham-website")[:20]



['about_living-the-mission_campus-ministry_catholic-life_ministry-of-music.md',
 'academics_centers-and-institutes_center-for-ethics-education_academic-programs_advanced-certificate-in-health-care-ethics_courses.md',
 'graduate-school-of-arts-and-sciences_student-resources_professional-development_preparing-future-faculty-program_resources-for-participants.md',
 'academics_departments_psychology_graduate-program_phd-in-applied-developmental-psychology_alumni-profiles.md',
 'academics_departments_african--african-american-studies_faculty.md',
 'resources_policies_undergraduate-academic-integrity-policy_student-resources.md',
 'graduate-school-of-social-service_faculty_full-time-faculty-profiles_shirley-gatenio-gabel.md',
 'school-of-law_faculty_directory_full-time_susan-block-lieb.md',
 'summer-session_registration-and-housing_policies-and-procedures.md',
 'academics_departments_biological-sciences_faculty-and-instructional-staff_kaoutsar-nasrallah.md',
 'summer-session_pre-college-prog

In [6]:
import os
os.listdir("/Users/purtilalan/Downloads/fordham-website")[:20]


['about_living-the-mission_campus-ministry_catholic-life_ministry-of-music.md',
 'academics_centers-and-institutes_center-for-ethics-education_academic-programs_advanced-certificate-in-health-care-ethics_courses.md',
 'graduate-school-of-arts-and-sciences_student-resources_professional-development_preparing-future-faculty-program_resources-for-participants.md',
 'academics_departments_psychology_graduate-program_phd-in-applied-developmental-psychology_alumni-profiles.md',
 'academics_departments_african--african-american-studies_faculty.md',
 'resources_policies_undergraduate-academic-integrity-policy_student-resources.md',
 'graduate-school-of-social-service_faculty_full-time-faculty-profiles_shirley-gatenio-gabel.md',
 'school-of-law_faculty_directory_full-time_susan-block-lieb.md',
 'summer-session_registration-and-housing_policies-and-procedures.md',
 'academics_departments_biological-sciences_faculty-and-instructional-staff_kaoutsar-nasrallah.md',
 'summer-session_pre-college-prog

---

# 2. Chunk the Documents

Some of the pages could be too long to embed as a single unit. Down the line, the pages may be too long to stuff into the LLM's prompt during the generation step. As such, most of the RAG systems will break down big documents into into smaller **chunks**.

> 📚 **TERM: Chunking**  
> Splitting documents into smaller, self-contained pieces for embedding and retrieval. The goal is chunks that are small enough to be specific, but large enough to be meaningful.

Your task: **write a function that splits each document into chunks.**

Things to think about:
- What's a reasonable chunk size? (Think about what fits in a prompt vs. what's too vague)
- Should you split on sentences? Paragraphs? A fixed character/word count?
- Should chunks overlap? What happens if an answer spans two chunks?
- How do you keep track of which document each chunk came from? You may need that information down the line.

In [7]:
# Placeholder for your implementation
import pandas as pd

# ---- Chunking Function ----
def chunk_text(text, chunk_size=500, overlap=100):
    """
    Splits text into overlapping chunks.
    
    chunk_size: number of characters per chunk
    overlap: characters shared between consecutive chunks
    """
    
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap   # move forward with overlap
    
    return chunks


# ---- Apply Chunking to ALL Documents ----
chunked_data = []

for _, row in df.iterrows():
    text_chunks = chunk_text(row["content"], chunk_size=500, overlap=100)
    
    for i, chunk in enumerate(text_chunks):
        chunked_data.append({
            "filename": row["filename"],
            "url": row["url"],
            "chunk_id": i,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunked_data)

print("Total chunks created:", len(chunks_df))
chunks_df.head()



Total chunks created: 104463


,filename,url,chunk_id,text
0,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,0,\n# Ministry of Music\n\n\nFordham offers each...
1,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,1,\nThe Fordham University Schola Cantorum is a ...
2,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,2,"century liturgical music, and music of the wor..."
3,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,3,uate students receive ensemble credit.\n\n**Di...
4,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,4,"oly Week Liturgies, and Baccalaureate Mass.\n\..."


In [8]:
chunks_df["text"] = chunks_df["text"].str.replace(r"\n+", " ", regex=True)
chunks_df["text"] = chunks_df["text"].str.strip()


---

# 3. Embed the Chunks

Now we need to turn each chunk into a vector so we can search over them. You've done this before in Lecture 4.

Your task: **embed all chunks using an embedding model.**

Tips:
- You could use a local model, or API model. What are the tradeoffs?
- This will take a while if you do it serially. You might want to use async/batch.
- Once you've created your embeddings, you may want to save them to disk so you don't have to redo this step every time
- You'll need to embed queries with the **same model** at search time

In [10]:
from sentence_transformers import SentenceTransformer
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)

# Reduce token length to lower memory use (HUGE win)
model.max_seq_length = 256   # try 256 (or 128 if still crashing)

texts = chunks_df["text"].tolist()
print("Total chunks to embed:", len(texts))

embeddings = model.encode(
    texts,
    batch_size=16,              # start at 16 on MPS
    show_progress_bar=True,
    normalize_embeddings=True
)

chunks_df["embedding"] = list(embeddings)
print("Embeddings completed.")

/Users/purtilalan/ai-engineering-fordham/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1489.21it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks to embed: 104463


Batches: 100%|██████████| 6529/6529 [50:58<00:00,  2.13it/s]  


Embeddings completed.


In [11]:
import numpy as np

# 1) Confirm count matches
print("Rows:", len(chunks_df))
print("Embeddings:", chunks_df["embedding"].notna().sum())

# 2) Confirm embedding dimension (should be 768 for bge-base)
first = chunks_df["embedding"].iloc[0]
print("First embedding shape:", np.array(first).shape)

# 3) Quick similarity test (optional)
q = "return policy for shoes"
q_emb = model.encode([q], normalize_embeddings=True)[0]
doc0 = chunks_df["embedding"].iloc[0]
print("Cosine similarity with first chunk:", float(np.dot(q_emb, doc0)))

Rows: 104463
Embeddings: 104463
First embedding shape: (768,)
Cosine similarity with first chunk: 0.3677014112472534


In [12]:
import numpy as np

emb = np.vstack(chunks_df["embedding"].to_numpy())
np.save("bge_embeddings.npy", emb)

chunks_df.drop(columns=["embedding"]).to_pickle("chunks_no_emb.pkl")

print("Saved: bge_embeddings.npy and chunks_no_emb.pkl")

Saved: bge_embeddings.npy and chunks_no_emb.pkl


In [13]:
import numpy as np
import pandas as pd

chunks_df = pd.read_pickle("chunks_no_emb.pkl")
emb = np.load("bge_embeddings.npy")

In [4]:
len(chunks_df)


104463

In [5]:
import pandas as pd

def chunk_text(text, chunk_size=1200, overlap=150):
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# rebuild chunks_df
chunked_data = []

for _, row in df.iterrows():
    text_chunks = chunk_text(row["content"], chunk_size=1200, overlap=150)
    for i, chunk in enumerate(text_chunks):
        chunked_data.append({
            "filename": row["filename"],
            "url": row["url"],
            "chunk_id": i,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunked_data)

print("Total chunks created:", len(chunks_df))
chunks_df.head()


Total chunks created: 42772


,filename,url,chunk_id,text
0,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,0,\n# Ministry of Music\n\n\nFordham offers each...
1,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,1,"Holy Week Liturgies, and the Baccalaureate Mas..."
2,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,2,orary liturgical music. An audition with the d...
3,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,3,"opolitan area, including Carnegie Hall, the Ro..."
4,about_living-the-mission_campus-ministry_catho...,https://www.fordham.edu/about/living-the-missi...,4,"nner, Baccalaureate, Commencement, and more)\n..."


In [6]:
chunks_df["text"].str.len().describe()


count    42772.000000
mean      1046.919083
std        314.375586
min          1.000000
25%       1184.750000
50%       1200.000000
75%       1200.000000
max       1200.000000
Name: text, dtype: float64

---

# 4. Retrieve

Now build the **R** in RAG. Given a user's question, find the most relevant chunks.

Your task: **write a retrieval function that takes a question and returns the most relevant chunks.**

Tips:
- You can use lexical or semantic search or both!
- How many chunks should you retrieve? Too few and you might miss the answer; too many and you'll overwhelm the LLM (and pay more tokens)
- Try a few test questions and eyeball whether the retrieved chunks are relevant
- Try a few questions and see what comes back. For example:
  - "What programs does the Gabelli School of Business offer?"
  - "How do I apply for financial aid?"
  - "Where is Fordham's campus?"

In [14]:
# Your implementation here
import re
import numpy as np
from sentence_transformers import SentenceTransformer

# --- Optional BM25 (lexical) setup ---
# If you want lexical search too:
# uv add rank-bm25
try:
    from rank_bm25 import BM25Okapi
    _HAS_BM25 = True
except ImportError:
    _HAS_BM25 = False

def _tokenize(text: str) -> list[str]:
    # simple, fast tokenizer
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if len(t) > 1]

def build_bm25_index(chunks_df, text_col="text"):
    """Build BM25 index once (do this once per notebook run)."""
    if not _HAS_BM25:
        raise ImportError("rank-bm25 not installed. Run: uv add rank-bm25")
    corpus_tokens = [_tokenize(t) for t in chunks_df[text_col].astype(str).tolist()]
    return BM25Okapi(corpus_tokens)

def retrieve(
    question: str,
    chunks_df,
    embeddings: np.ndarray,
    model: SentenceTransformer,
    *,
    k: int = 8,
    method: str = "hybrid",          # "semantic" | "lexical" | "hybrid"
    text_col: str = "text",
    bm25=None,                       # pass BM25Okapi if using lexical/hybrid
    alpha: float = 0.6               # hybrid weight: semantic vs lexical (0..1)
):
    """
    Returns top-k most relevant chunks for a question.
    - semantic: cosine similarity over normalized embeddings
    - lexical: BM25 keyword match
    - hybrid: weighted blend of normalized semantic + lexical scores
    """
    method = method.lower()
    if method not in {"semantic", "lexical", "hybrid"}:
        raise ValueError("method must be one of: semantic, lexical, hybrid")

    # --- Semantic scores (cosine if embeddings are normalized) ---
    q_emb = model.encode([question], normalize_embeddings=True)[0]
    sem_scores = embeddings @ q_emb  # shape: (N,)

    if method == "semantic":
        idx = np.argsort(sem_scores)[-k:][::-1]
        out = chunks_df.iloc[idx].copy()
        out["score"] = sem_scores[idx]
        out["method"] = "semantic"
        return out.reset_index(drop=True)

    # --- Lexical (BM25) scores ---
    if bm25 is None:
        raise ValueError("bm25 index required for lexical/hybrid. Build it with build_bm25_index().")

    q_tokens = _tokenize(question)
    lex_scores = np.array(bm25.get_scores(q_tokens), dtype=np.float32)

    if method == "lexical":
        idx = np.argsort(lex_scores)[-k:][::-1]
        out = chunks_df.iloc[idx].copy()
        out["score"] = lex_scores[idx]
        out["method"] = "lexical"
        return out.reset_index(drop=True)

    # --- Hybrid: normalize both to 0..1 and blend ---
    # Avoid divide-by-zero when all scores are equal
    def _minmax(x):
        x = x.astype(np.float32)
        rng = x.max() - x.min()
        return (x - x.min()) / (rng + 1e-8)

    sem_n = _minmax(sem_scores)
    lex_n = _minmax(lex_scores)

    hybrid_scores = alpha * sem_n + (1 - alpha) * lex_n
    idx = np.argsort(hybrid_scores)[-k:][::-1]

    out = chunks_df.iloc[idx].copy()
    out["score"] = hybrid_scores[idx]
    out["score_sem"] = sem_scores[idx]
    out["score_lex"] = lex_scores[idx]
    out["method"] = "hybrid"
    return out.reset_index(drop=True)

In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import torch

chunks_df = pd.read_pickle("chunks_no_emb.pkl")
embeddings = np.load("bge_embeddings.npy")

device = "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)

results = retrieve(
    "What programs does the Gabelli School of Business offer?",
    chunks_df,
    embeddings,
    model,
    k=8,
    method="semantic"
)

results[["score", "text"]].head(8)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2123.77it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,score,text
0,0.830738,Sales](/media/home/schools/gabelli-school-of-b...
1,0.827291,ions Gabelli School of Business
2,0.807462,t a world-class business school with practical...
3,0.802583,# Gabelli School of Business ## Graduate Busin...
4,0.802583,# Gabelli School of Business ## Graduate Busin...
5,0.801187,arketing Fordham University Gabelli School of ...
6,0.788193,"ils on the curriculum for each one, use the li..."
7,0.786806,# Gabelli Honors Programs ![Honors Opportuniti...


In [16]:
import re
import numpy as np

# --- BM25 import ---
try:
    from rank_bm25 import BM25Okapi
    _HAS_BM25 = True
except ImportError:
    _HAS_BM25 = False
    print("⚠️ rank-bm25 not installed. Run: uv add rank-bm25")

# --- tokenizer ---
def _tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if len(t) > 1]

# --- BM25 builder ---
def build_bm25_index(chunks_df, text_col="text"):
    if not _HAS_BM25:
        raise ImportError("rank-bm25 not installed. Run: uv add rank-bm25")
    corpus_tokens = [
        _tokenize(t) for t in chunks_df[text_col].astype(str).tolist()
    ]
    return BM25Okapi(corpus_tokens)

In [17]:
bm25 = build_bm25_index(chunks_df, text_col="text")
print("BM25 built.")

BM25 built.


---

# 5. Generate

Now build the **G** in RAG. Take the retrieved chunks and pass them to an LLM along with the user's question.

Your task: **write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.**

Tips:
- How should you structure the prompt? The LLM needs to know: (1) what is the context of the application, (2) what is the question, (3) what it should include in its answer
- What should the LLM do if the context doesn't contain the answer?
- Start with a cheap model; try a better one when you've figured out the pipeline

In [31]:
from openai import OpenAI

client = OpenAI()

def generate_answer(
    question: str,
    retrieved_df,
    *,
    text_col: str = "text",
    model: str = "gpt-4o-mini",
    max_chunks: int = 6
):
    """
    Generate an answer using retrieved context (RAG).

    Inputs:
        question: user question
        retrieved_df: dataframe returned by retrieve()
    """

    # ---- 1. Build context block ----
    contexts = retrieved_df[text_col].astype(str).tolist()[:max_chunks]

    context_text = "\n\n---\n\n".join(
    f"[Source {i+1}]\n{c}" for i, c in enumerate(contexts)
)
    

    # ---- 2. Build prompt ----
    system_prompt = (
        "You are a helpful assistant for answering questions about Fordham University. "
        "Use ONLY the provided context to answer. "
        "If the answer is not in the context, say you don't know."
    )

    user_prompt = f"""
Answer the question using the context below.

Context:
{context_text}

Question:
{question}

Instructions:
- Be concise and factual
- Use only the provided context
- If the context is insufficient, say: "I don't have enough information to answer this."
- Cite the source number(s) used (e.g., [Source 2])

Answer:
"""

    # ---- 3. Call LLM ----
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    return resp.choices[0].message.content.strip()

In [19]:
# Placeholder for your implementation
question = "How do I apply for financial aid?"

# retrieve
retrieved = retrieve(
    question,
    chunks_df,
    embeddings,
    model,
    k=8,
    method="hybrid",
    bm25=bm25
)

# generate
answer = generate_answer(question, retrieved)

print(answer)

To apply for financial aid, you need to file the Free Application for Federal Student Aid (FAFSA). If you are a New York state resident and list at least one New York school on your FAFSA, you will receive a link to complete the TAP application. It is important to apply every year, even if you think you may not be eligible for aid.


---

# 6. Wire everything together

Combine the previous steps into a simple function that takes in a question and returns an answer.

Your task: **write a `rag(question)` function that retrieves relevant chunks and generates an answer.**

In [20]:
# Placeholder for your implementation
def rag(
    question: str,
    *,
    k: int = 8,
    method: str = "hybrid",   # "semantic" | "lexical" | "hybrid"
    max_chunks: int = 6,
    alpha: float = 0.6,
    debug: bool = False
):
    """
    End-to-end RAG pipeline.

    Steps:
    1) Retrieve relevant chunks
    2) Generate grounded answer
    """

    # ---- Retrieve ----
    retrieved = retrieve(
        question,
        chunks_df,
        embeddings,
        model,
        k=k,
        method=method,
        bm25=bm25 if method in ("hybrid", "lexical") else None,
        alpha=alpha,
    )

    # ---- Generate ----
    answer = generate_answer(
        question,
        retrieved,
        max_chunks=max_chunks
    )

    if debug:
        return {
            "question": question,
            "answer": answer,
            "retrieved": retrieved
        }

    return answer

In [21]:
print(rag("What programs does the Gabelli School of Business offer?"))

The Gabelli School of Business offers three types of M.B.A. programs (full-time, professional, and executive M.B.A.), 12 M.S. programs (two offered online), and two doctoral programs (Ph.D. and Doctor of Professional Studies).


In [22]:
questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Where is Fordham's campus?"
]

for q in questions:
    print("\nQUESTION:", q)
    print(rag(q))


QUESTION: What programs does the Gabelli School of Business offer?
The Gabelli School of Business offers three types of M.B.A. programs (full-time, professional, and executive M.B.A.), 12 M.S. programs (two offered online), and two doctoral programs (Ph.D. and Doctor of Professional Studies).

QUESTION: How do I apply for financial aid?
To apply for financial aid at Fordham University, you need to file the Free Application for Federal Student Aid (FAFSA). If you are a New York state resident and list at least one New York school on your FAFSA, you will receive a link to complete the TAP application.

QUESTION: Where is Fordham's campus?
Fordham's campus is located in New York City, with one campus at Lincoln Center and another at Rose Hill in the Bronx.


---

# 7. Evaluate, experiment and improve

Your RAG system works — but there's always room to make it better. 

Your task: **evaluate, experiment, and improve your system**

Tips:
- How do you know that your system is working or that your changes are improving it?
- Try different questions — where does it do well? Where does it struggle?
- Adjust the number of retrieved chunks — what happens with more or fewer?
- Try different chunking strategies — bigger chunks? Smaller? Overlap?
- Try a different embedding model — does it change retrieval quality?
- Improve the prompt — can you get better, more concise answers?
- Add source attribution — can the system tell the user which pages the answer came from?

In [23]:
test_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Where is Fordham's campus located?",
    "What is the tuition for MBA students?",
    "Does Fordham offer undergraduate business degrees?"
]

for q in test_questions:
    print("\nQUESTION:", q)
    print(rag(q))


QUESTION: What programs does the Gabelli School of Business offer?
The Gabelli School of Business offers three types of M.B.A. programs (full-time, professional, and executive M.B.A.), 12 M.S. programs (two offered online), and two doctoral programs (Ph.D. and Doctor of Professional Studies).

QUESTION: How do I apply for financial aid?
To apply for financial aid at Fordham University, you need to file the Free Application for Federal Student Aid (FAFSA). If you are a New York state resident and list at least one New York school on your FAFSA, you will receive a link to complete the TAP application.

QUESTION: Where is Fordham's campus located?
Fordham's Westchester campus is located at 400 Westchester Avenue, West Harrison, NY 10604. The Lincoln Center campus is located at 113 W. 60th Street (at Columbus Avenue) in New York City. The Rose Hill campus is located in the Bronx.

QUESTION: What is the tuition for MBA students?
The flat-rate tuition for the Executive MBA program is $114,0

In [24]:
def evaluate_retrieval(question, k=8):
    out = rag(question, k=k, debug=True)
    retrieved = out["retrieved"]

    print("QUESTION:", question)
    print("\nTop retrieved chunks:")
    for i, row in retrieved.head(5).iterrows():
        snippet = row["text"][:180].replace("\n", " ")
        print(f"{i+1}. score={row['score']:.3f} | {snippet}...")

In [25]:
evaluate_retrieval("How do I apply for financial aid?")

QUESTION: How do I apply for financial aid?

Top retrieved chunks:
1. score=0.925 | within two to four weeks of the date you submitted the FAFSA for processing. - **How do I apply for New York state aid?**When filing the FAFSA online, all New York state residents ...
2. score=0.921 | tudent-financial-services/student--parent-portals/) ## How to Apply for Scholarships and Financial Aid Wherever you are in your academic journey, we can help you maximize your oppo...
3. score=0.921 | tudent-financial-services/student--parent-portals/) ## How to Apply for Scholarships and Financial Aid Wherever you are in your academic journey, we can help you maximize your oppo...
4. score=0.921 | tudent-financial-services/student--parent-portals/) ## How to Apply for Scholarships and Financial Aid Wherever you are in your academic journey, we can help you maximize your oppo...
5. score=0.921 | tudent-financial-services/student--parent-portals/) ## How to Apply for Scholarships and Financial Aid Wherever 

In [26]:
for k in [4, 6, 8, 12]:
    print(f"\n===== k={k} =====")
    print(rag("How do I apply for financial aid?", k=k))


===== k=4 =====
To apply for financial aid at Fordham University, you can learn how to apply through the student financial services section of their website. They provide resources to help maximize your opportunities for receiving financial aid.

===== k=6 =====
To apply for financial aid at Fordham University, you need to file the Free Application for Federal Student Aid (FAFSA). If you are a New York state resident and list at least one New York school on your FAFSA, you will receive a link to complete the TAP application. It is important to apply every year, even if you think you may not be eligible for aid.

===== k=8 =====
To apply for financial aid, you need to file the Free Application for Federal Student Aid (FAFSA). If you are a New York state resident and list at least one New York school on your FAFSA, you will receive a link to complete the TAP application. Additionally, it is important to apply every year for financial aid.

===== k=12 =====
To apply for financial aid, yo

In [27]:
models_to_try = [
    "BAAI/bge-base-en-v1.5",
    "sentence-transformers/all-MiniLM-L6-v2"
]

In [28]:
system_prompt = (
    "You are a careful university assistant. "
    "Answer ONLY using the provided context. "
    "If the answer is not in the context, say you don't know. "
    "Do not make up information."
)

---

# 8. (Optional) Make it an app

So far your RAG system lives inside a notebook. That's great for development — but nobody is going to use your Jupyter notebook to ask questions about Fordham. Let's turn it into a real web app.

> 📚 **TERM: Streamlit**  
> A Python library that turns plain Python scripts into interactive web apps. You write Python — no HTML, CSS, or JavaScript — and Streamlit renders it as a web page with inputs, buttons, and formatted output. It's the fastest way to go from "I have a function" to "I have a web app."

Your task: **create a Streamlit app that lets a user type a question about Fordham and get an answer from your RAG system.**

To get started:
- Install it: `uv pip install streamlit` 
- A Streamlit app is just a `.py` file (not a notebook). Create something like `fordham_rag_app.py`
- Run it: `streamlit run scripts/fordham_rag_app.py` — this opens a browser tab with your app

Tips:
- Check out the [Streamlit docs](https://docs.streamlit.io/) — the "Get started" tutorial is very short
- Your best bet is to vibecode your way to this. You'll be surprised how fast you can get it up and running

---

# Summary

## What You Built

| Step | What You Did | What It Does |
|------|-------------|-------------|
| **Load** | Read 9,500+ Fordham web pages | Get raw content |
| **Chunk** | Split pages into smaller pieces | Make content searchable and promptable |
| **Embed** | Turn chunks into vectors | Enable semantic search |
| **Retrieve** | Find relevant chunks for a question | The **R** in RAG |
| **Generate** | Ask an LLM to answer using the chunks | The **G** in RAG |
| **RAG** | Wire it all together | Question in, answer out |

## The Big Picture

RAG is one of the most common patterns in AI engineering today. What you built here is the same core architecture behind tools like ChatGPT with search, Perplexity, enterprise Q&A bots, and more. The details get more sophisticated (vector databases, reranking, query rewriting, evaluation) but the pattern is the same:

**Find relevant stuff → give it to an LLM → get an answer.**

You can just build things.

Embedding model used: BAAI/bge-base-en-v1.5 (SentenceTransformers, runs on MPS if available)
LLM used for generation: gpt-4o-mini (OpenAI Chat Completions)
LLM used for evaluation (judge): gpt-4o-mini 
Saved artifacts: bge_embeddings.npy, chunks_no_emb.pkl
How to start the Streamlit app: streamlit run scripts/fordham_rag_app.py
Any API keys or env vars needed: OPENAI_API_KEY stored in .env file
Anything else I should know:
First run may download the embedding model from HuggingFace (~80MB).
App expects bge_embeddings.npy and chunks_no_emb.pkl to be in the project root.
If MPS runs out of memory, use CPU by setting device to "cpu" in the SentenceTransformer load.

Bonus: Experiment and Improve-
Tested different values of top-k retrieval and max_chunks to balance relevance vs prompt length.
Improved the prompt to reduce false “not enough info” answers and require citations.
Added source display + confidence score to improve transparency and debugging.